## PDF loader

In [21]:
## Load pdf document

from langchain_community.document_loaders import PyPDFLoader

file_path = "D:\\GenAI\\Agentic AI\\Test_Projects\\RAG-Project\\data\\PDF\\Rainbow-Bazaar-Return-Refund-&-Cancellation-Policy.pdf"
pdf_loader = PyPDFLoader(file_path)

pdf_docs = pdf_loader.load()

## Chunking

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter


In [23]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

In [24]:
pdf_chunks = text_splitter.split_documents(pdf_docs)

In [25]:
print(len(pdf_chunks))

9


In [26]:
print(pdf_chunks[0].page_content[:1000])

Return, Refund and Cancellation Policy 
1.1. Refunds, Cancellations and Returns of Goods 
Please note that you can submit complaints with the Order and product sold via 
rainbowbazaar.shop and we shall process the refund to you within 30 days from date of 
receipt of complaint. Once we issue your refund, it takes additional time for your financial 
institution to make funds available in your account, which can vary from 2-10 days from 
the date of refund processing. All orders are manually processed on the Website and sent 
for shipment as soon as they are placed. During this process we incur some irreversible 
fees. Therefore, while we understand that orders might need to be changed sometimes, we 
are unable to do it free of charge after a certain point. We strictly adhere to the following 
cancellation policy: 
• If you cancel your order BEFORE it has been shipped, you will not be charged any cancellation 
fee;


## Embedding

In [27]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [28]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Creating a vector Store

In [29]:
from langchain_community.vectorstores import Chroma

In [30]:
vector_store = Chroma.from_documents(documents = pdf_chunks,embedding = embedding_model, persist_directory = "D:\\GenAI\\Agentic AI\\Test_Projects\\RAG-Project\\vector-store\\chroma_db")

vector_store.persist()

Lets verify by reloading the vector store

In [31]:
print("Total Vectors in Store: ", vector_store._collection.count())

Total Vectors in Store:  36


# Querying

### Create a retriever

In [32]:
retriever = vector_store.as_retriever(search_kwargs={"k":3})

### Create an LLM

In [35]:
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

True

In [36]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

### Build RAG chain

In [37]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from langchain_core.documents import Document


In [38]:
#### Helper function to format documents into a string 

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in docs])

In [39]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that helps people find information based on the PDF only."),
    ("human", "Context: {context}\n\nQuestion: {question}\n\nAnswer in a concise manner.")
])

In [40]:
qa_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [41]:
query = "What are the cancellation charges?"

In [42]:
response = qa_chain.invoke(query)

print("Response: ", response)

Response:  The policy states that for cancellations **after 7 days but before 15 days** from the order date, a charge is applied equal to **the number of days × 1 % of the order total**.

- Cancel on day 8 → 8 % of the order amount is deducted.  
- Cancel on day 9 → 9 % of the order amount is deducted.  
- …and so on up to day 14 (14 % charge).

Cancellations made within the first 7 days are not subject to this charge.
